# head

> Linear classifier over the stored embeddings. The Train button fits it; recognition uses it while it still matches the library.


In [ ]:
#| default_exp head


In [ ]:
#| hide
from nbdev.showdoc import *


Each taught sound keeps up to twelve 768-d embeddings. Until you train, a new clip is the nearest of those examples (max cosine). Training fits one linear layer, `768 → names`, on every stored vector and writes `head.npz` next to the database.

The cosine threshold is unchanged: a name is accepted only when its best stored example still clears `mincos`. Teaching, forgetting, or adding an example changes the fingerprint, and recognition goes back to nearest-example until the next Train.

`fit_head` is what the app runs. It is NumPy, so the server does not import PyTorch. The fastai notebook writes the same file.


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import hashlib
from pathlib import Path

import numpy as np

from nadakit.core import data_dir
from nadakit.engine import dim as EMBEDDING_SIZE


In [ ]:
#| export
def head_path() -> Path:
    """`head.npz` beside the sound database."""
    return data_dir() / "head.npz"


In [ ]:
#| export
def vectors(samples: bytes, dim: int = EMBEDDING_SIZE) -> np.ndarray:
    """`(n, dim)` float32 embeddings stored on a sound, or a `(0, dim)` array."""
    raw = np.frombuffer(bytes(samples or b""), dtype=np.float32)
    if not raw.size or raw.size % dim:
        return np.zeros((0, dim), np.float32)
    return raw.reshape(-1, dim).copy()


In [ ]:
#| export
def fingerprint(rows) -> str:
    """Changes when any taught id or its embedding blob changes."""
    digest = hashlib.sha1()
    for row in sorted(rows, key=lambda s: int(s.id)):
        digest.update(int(row.id).to_bytes(8, "little", signed=True))
        digest.update(bytes(row.samples or b""))
    return digest.hexdigest()


In [ ]:
#| export
def design(rows):
    """Stacked embeddings, class index per row, sound ids, and the library fingerprint.

    `None` when fewer than two sounds have an example.
    """
    rows = [s for s in rows if len(vectors(s.samples))]
    if len(rows) < 2:
        return None
    chunks = [vectors(s.samples) for s in rows]
    X = np.vstack(chunks).astype(np.float32, copy=False)
    y = np.concatenate([np.full(len(v), i, np.int64) for i, v in enumerate(chunks)])
    ids = np.asarray([s.id for s in rows], np.int64)
    return X, y, ids, fingerprint(rows)


In [ ]:
#| export
def fit_head(X, y, *, l2: float = 0.05, steps: int = 400, lr: float = 0.2):
    """L2-regularized softmax regression. Rows of `W` start at the class means.

    `X` is `(n, 768)`, `y` is `0 .. n_classes-1`. Returns `W` `(classes, 768)` and `b` `(classes,)`.
    """
    X = np.asarray(X, np.float64)
    y = np.asarray(y, np.int64)
    n, d = X.shape
    c = int(y.max()) + 1
    W = np.zeros((c, d))
    for k in range(c):
        take = X[y == k]
        if len(take):
            W[k] = take.mean(0)
    b = np.zeros(c)
    mW, vW = np.zeros_like(W), np.zeros_like(W)
    mb, vb = np.zeros_like(b), np.zeros_like(b)
    b1, b2, eps = 0.9, 0.999, 1e-8
    Y = np.eye(c)[y]
    for t in range(1, steps + 1):
        logits = X @ W.T + b
        logits -= logits.max(1, keepdims=True)
        P = np.exp(logits)
        P /= P.sum(1, keepdims=True)
        G = (P - Y) / n
        gW = G.T @ X + l2 * W
        gb = G.sum(0)
        mW = b1 * mW + (1 - b1) * gW
        vW = b2 * vW + (1 - b2) * (gW * gW)
        mb = b1 * mb + (1 - b1) * gb
        vb = b2 * vb + (1 - b2) * (gb * gb)
        W -= lr * (mW / (1 - b1**t)) / (np.sqrt(vW / (1 - b2**t)) + eps)
        b -= lr * (mb / (1 - b1**t)) / (np.sqrt(vb / (1 - b2**t)) + eps)
    return W.astype(np.float32), b.astype(np.float32)


In [ ]:
#| export
def save_head(W, b, ids, fp: str, path: Path | None = None) -> Path:
    """Write the head the recognizer loads. Replaces any previous file."""
    path = Path(path) if path else head_path()
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("wb") as handle:
        np.savez(
            handle,
            W=np.asarray(W, np.float32),
            b=np.asarray(b, np.float32),
            ids=np.asarray(ids, np.int64),
            fp=np.asarray(fp),
        )
    tmp.replace(path)
    return path


In [ ]:
#| export
def load_head(path: Path | None = None):
    """The saved head, or `None` when no file has been written."""
    path = Path(path) if path else head_path()
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as z:
        return dict(
            W=np.asarray(z["W"], np.float32),
            b=np.asarray(z["b"], np.float32),
            ids=np.asarray(z["ids"], np.int64),
            fp=str(z["fp"]),
        )


In [ ]:
#| export
def active_head(rows, path: Path | None = None):
    """The head when its fingerprint still matches `rows`, else `None`."""
    head = load_head(path)
    if head is None or not rows:
        return None
    if head["fp"] != fingerprint(rows):
        return None
    if {int(i) for i in head["ids"]} != {int(s.id) for s in rows}:
        return None
    return head


In [ ]:
#| export
def head_logits(head, embedding, ids) -> np.ndarray:
    """One logit per id. An id the head was not trained on scores far below the others."""
    e = np.asarray(embedding, np.float32).reshape(-1)
    index = {int(i): k for k, i in enumerate(np.asarray(head["ids"]).tolist())}
    W, b = head["W"], head["b"]
    out = np.full(len(ids), -1e9, np.float32)
    known = [(n, index[int(i)]) for n, i in enumerate(ids) if int(i) in index]
    if known:
        ns, ks = zip(*known)
        out[list(ns)] = W[list(ks)] @ e + b[list(ks)]
    return out


In [ ]:
#| export
def score_library(embedding, rows, universe=None, path=None):
    """Rank `rows` by the head when `universe` still matches it, else by max cosine.

    `universe` is the whole library. The keyboard ranks only the sounds that have a key,
    and the head was fitted to every sound, so the fingerprint is taken on `universe`.
    Each result is `id`, `cos` (max cosine, the acceptance check), and `share`.
    """
    universe = list(rows if universe is None else universe)
    rows = [s for s in rows if len(vectors(s.samples))]
    universe = [s for s in universe if len(vectors(s.samples))]
    if not rows:
        return []
    e = np.asarray(embedding, np.float32).reshape(-1)
    blocks = [vectors(s.samples) for s in rows]
    cos = np.array([float((v @ e).max()) for v in blocks], np.float64)
    head = active_head(universe, path)
    if head is None:
        score = cos
        w = np.exp((score - score.max()) * 40)
    else:
        score = head_logits(head, e, [s.id for s in rows]).astype(np.float64)
        w = np.exp(score - score.max())
    w = w / w.sum()
    order = np.argsort(-score, kind="stable")
    return [
        dict(id=int(rows[i].id), cos=float(cos[i]), share=float(w[i]), score=float(score[i]))
        for i in order
    ]


In [ ]:
#| export
def train_message(n_examples: int, n_sounds: int) -> str:
    ex = "example" if n_examples == 1 else "examples"
    so = "sound" if n_sounds == 1 else "sounds"
    return f"Trained on {n_examples} {ex} of {n_sounds} {so}. Recognize uses that head now."


In [ ]:
#| export
def train_library(rows, path: Path | None = None):
    """Fit `fit_head` on every example and save it. Returns `(message, failed)`."""
    built = design(rows)
    if built is None:
        return "Teach at least two different sounds before training.", True
    X, y, ids, fp = built
    W, b = fit_head(X, y)
    save_head(W, b, ids, fp, path)
    return train_message(int(y.size), len(ids)), False


In [ ]:
#| hide
class _Row:
    def __init__(self, id, samples):
        self.id, self.samples = id, samples

def _unit(rng, x):
    x = np.asarray(x, np.float64)
    return x / np.linalg.norm(x)

def _check():
    rng = np.random.default_rng(0)
    a = _unit(rng, rng.normal(size=EMBEDDING_SIZE))
    b = _unit(rng, a + 0.8 * rng.normal(size=EMBEDDING_SIZE))
    A = np.stack([_unit(rng, a + 0.02 * rng.normal(size=EMBEDDING_SIZE)) for _ in range(8)])
    B = np.vstack([
        np.stack([_unit(rng, b + 0.02 * rng.normal(size=EMBEDDING_SIZE)) for _ in range(8)]),
        _unit(rng, a),
    ])
    q = _unit(rng, a + 0.01 * rng.normal(size=EMBEDDING_SIZE)).astype(np.float32)
    assert float((B @ q).max()) > float((A @ q).max())
    rows = [_Row(1, A.astype(np.float32).tobytes()), _Row(2, B.astype(np.float32).tobytes())]
    msg, err = train_library(rows, path := __import__("tempfile").NamedTemporaryFile(suffix=".npz", delete=False).name)
    assert not err, msg
    scored = score_library(q, rows, rows, path)
    assert scored[0]["id"] == 1
    assert active_head(rows, path) is not None
    rows[0].samples += b"\x00\x00\x00\x00"
    assert active_head(rows, path) is None
    Path(path).unlink(missing_ok=True)

_check()
